In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Najafgarh_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,368.0,NaN,190.0,97.0,194.0,177.0,61.0,50.0,73.0,NaN,288.0,275.0
1,2,285.0,200.0,85.0,145.0,197.0,111.0,65.0,54.0,53.0,125.0,285.0,262.0
2,3,338.0,132.0,120.0,182.0,259.0,118.0,66.0,51.0,52.0,109.0,401.0,227.0
3,4,334.0,171.0,128.0,122.0,314.0,161.0,48.0,46.0,35.0,115.0,364.0,169.0
4,5,268.0,125.0,111.0,156.0,232.0,252.0,53.0,156.0,28.0,100.0,396.0,NaN
5,6,267.0,84.0,150.0,142.0,250.0,162.0,50.0,53.0,33.0,NaN,348.0,211.0
6,7,263.0,110.0,188.0,153.0,273.0,204.0,49.0,37.0,30.0,93.0,367.0,242.0
7,8,238.0,94.0,147.0,126.0,170.0,NaN,47.0,NaN,31.0,110.0,370.0,264.0
8,9,283.0,167.0,168.0,147.0,133.0,153.0,64.0,NaN,58.0,118.0,354.0,209.0
9,10,271.0,228.0,179.0,156.0,151.0,153.0,77.0,NaN,71.0,111.0,339.0,180.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,368.000000,155.066667,190.0,97.000000,194.000000,177.000000,61.000000,50.000000,73.000000,166.264706,288.000000,275.0
1,2,285.000000,200.000000,85.0,145.000000,197.000000,111.000000,65.000000,54.000000,53.000000,125.000000,285.000000,262.0
2,3,338.000000,132.000000,120.0,182.000000,259.000000,118.000000,66.000000,51.000000,52.000000,109.000000,401.000000,227.0
3,4,334.000000,171.000000,128.0,122.000000,165.617647,161.000000,48.000000,46.000000,35.000000,115.000000,364.000000,169.0
4,5,268.000000,125.000000,111.0,156.000000,232.000000,252.000000,53.000000,45.034483,28.000000,100.000000,396.000000,235.0
5,6,267.000000,84.000000,150.0,142.000000,250.000000,162.000000,50.000000,53.000000,33.000000,166.264706,348.000000,211.0
6,7,263.000000,110.000000,188.0,153.000000,165.617647,204.000000,49.000000,37.000000,30.000000,93.000000,367.000000,242.0
7,8,238.000000,94.000000,147.0,126.000000,170.000000,125.914286,47.000000,45.034483,31.000000,110.000000,370.000000,264.0
8,9,283.000000,167.000000,168.0,147.000000,133.000000,153.000000,64.000000,45.034483,58.000000,118.000000,354.000000,209.0
9,10,271.000000,228.000000,179.0,156.000000,151.000000,153.000000,77.000000,45.034483,71.000000,111.000000,339.000000,180.0
